# 02 · Activity-level evidence with X-CLIP (Step 3)

SigLIP (notebook 01) scores **single frames**, so it recognises objects and scenes but not motion: "opening" and "closing" a window looked identical. X-CLIP encodes **8 frames over 4 seconds** at once, so it can see what is happening. This notebook builds the activity curve **a(t)** and compares it with the frame curve **s(t)** using the same proposal and evaluation code.

**Session settings:** Accelerator **GPU T4 x2** · Internet **On**
**Inputs (Add Input → Datasets):** `charades-sta-vidsearch` (videos + annotations) and `vidsearch-features` (SigLIP features from notebook 01, for the comparison).

**Workflow:** run with `DEV = True` first (about 10 minutes, including a small model comparison), check the outputs, then set `DEV = False` and *Save Version → Save & Run All (Commit)*.

In [ ]:
# 1. Get the code from GitHub and run every unit test
from pathlib import Path
import sys, glob, subprocess

REPO_URL = "https://github.com/zankhana46/AI-Video-Investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo
sys.path.insert(0, "/tmp/repo")

for t in sorted(glob.glob("/tmp/repo/tests/test_*.py")):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, cwd="/tmp/repo")
    print(r.stdout.strip() or f"(no output from {Path(t).name})")
    if r.returncode:
        print(r.stderr[-2000:])
assert Path("/tmp/repo/vidsearch/xclip_features.py").exists(), "Step 3 files are not on GitHub yet: push them first"

In [ ]:
# 2. Find the attached datasets (handles both a packed videos.tar and Kaggle's auto-extracted folder)
from pathlib import Path
IN = Path("/kaggle/input")
print("attached:", sorted(p.name for p in IN.iterdir()))

ann = sorted(IN.rglob("val.jsonl"))
assert ann, "Attach the Charades data: Add Input -> Datasets -> charades-sta-vidsearch"
ANN_DIR = ann[0].parent

tars = [p for p in IN.rglob("videos.tar") if p.is_file()]
if tars:
    !tar -xf "{tars[0]}" -C /tmp
    VID_DIR = Path("/tmp/videos")
else:
    VID_DIR = next(IN.rglob("*.mp4")).parent
print("annotations:", ANN_DIR)
print("videos:", VID_DIR, "| count:", len(list(VID_DIR.glob("*.mp4"))))          # expect 1634

npz = [p for p in IN.rglob("*.npz") if not p.name.startswith("xclip_")]
FRAME_FEAT_DIR = npz[0].parent if npz else None
print("SigLIP features:", FRAME_FEAT_DIR or "NOT attached (Add Input -> vidsearch-features); frame comparison will be skipped")

In [ ]:
# 3. Settings
DEV = True                                   # True: 20 val + 20 test videos. False: everything.
MODEL = "microsoft/xclip-base-patch32"       # 8 frames per 4-s clip, the cheapest X-CLIP
COMPARE = ["microsoft/xclip-base-patch32",   # DEV only: which variant is worth the GPU time?
           "microsoft/xclip-base-patch16-zero-shot"]
COMPARE_VIDEOS = 40                          # val videos used for that comparison
CLIP_SEC, STRIDE_SEC = 4.0, 1.0              # 4-second clips, one every second
DEV_VIDEOS = 20

WORK = Path("/kaggle/working")
FEAT_OUT, RES_OUT = WORK / "features_activity", WORK / "results"
RES_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RES_OUT / "results_step3.csv"

In [ ]:
# 4. Load annotations and pick the videos
from vidsearch.curves import load_jsonl, videos_of

val_all = load_jsonl(ANN_DIR / "val.jsonl", "val")
test_all = load_jsonl(ANN_DIR / "test.jsonl", "test")
print("example record:", val_all[0])

if DEV:
    keep = set(videos_of(val_all)[:DEV_VIDEOS]) | set(videos_of(test_all)[:DEV_VIDEOS])
    val = [r for r in val_all if r["vid"] in keep]
    test = [r for r in test_all if r["vid"] in keep]
else:
    val, test = val_all, test_all
recs = val + test
assert len({r["qid"] for r in recs}) == len(recs), "query ids are not unique"

videos = [(v, VID_DIR / f"{v}.mp4") for v in videos_of(recs)]
missing = [v for v, p in videos if not p.exists()]
videos = [(v, p) for v, p in videos if p.exists()]
print(f"val: {len(val)} queries | test: {len(test)} queries | videos: {len(videos)} | missing files: {len(missing)}")

In [ ]:
# 5. Load X-CLIP and check our fast scoring against X-CLIP's own forward pass
import numpy as np, torch
from vidsearch.xclip_features import XClipScorer, load_video_clips

scorer = XClipScorer(MODEL)
print(f"{MODEL}: {scorer.frames_per_clip} frames per {CLIP_SEC:.0f}-s clip, {scorer.size}px, {scorer.dtype}")
print("pixel normalisation mean/std:", scorer.mean_list, scorer.std_list)

vid0, path0 = videos[0]
clips0 = load_video_clips(path0, scorer.frames_per_clip, CLIP_SEC, STRIDE_SEC, scorer.size)
qs0 = [r["query"] for r in recs if r["vid"] == vid0]
err, ref, mine = scorer.self_check(clips0["frames"], clips0["idx"], qs0)
print(f"video {vid0}: {clips0['n_sec']} s -> {len(clips0['starts'])} clips, frame grid {clips0['frames'].shape}")
print(f"max |fast - reference| = {err:.5f} ->",
      "OK, using the fast path" if not scorer.use_forward
      else "MISMATCH: falling back to X-CLIP's own forward (slower but correct). Tell Claude.")

In [ ]:
# 6. (DEV only) Compare X-CLIP variants on a few val videos before spending GPU time
if DEV and COMPARE:
    import pandas as pd
    from vidsearch.xclip_features import compare_models
    cmp_vids = set(videos_of(val_all)[:COMPARE_VIDEOS])
    cmp_recs = [r for r in val_all if r["vid"] in cmp_vids]
    print(f"comparing on {len(cmp_recs)} val queries from {len(cmp_vids)} videos (untuned settings)")
    cmp_table = pd.DataFrame(compare_models(COMPARE, cmp_recs, VID_DIR, CLIP_SEC, STRIDE_SEC))
    display(cmp_table)
    cmp_table.to_csv(RES_OUT / "dev_model_comparison.csv", index=False)

In [ ]:
# 7. Extract X-CLIP features for every video (the only GPU-heavy cell)
from vidsearch.xclip_features import extract, save_activity

vid_queries, query_text = {}, {}
for r in recs:
    vid_queries.setdefault(r["vid"], []).append(r["qid"])
    query_text[r["qid"]] = r["query"]

res = extract(scorer, videos, vid_queries, query_text, CLIP_SEC, STRIDE_SEC)

temb = scorer.encode_text([r["query"] for r in recs]).cpu().numpy().astype(np.float16)
text_emb = {r["qid"]: temb[i] for i, r in enumerate(recs)}
config = {"model": MODEL, "frames_per_clip": scorer.frames_per_clip, "clip_sec": CLIP_SEC,
          "stride_sec": STRIDE_SEC, "image_size": scorer.size,
          "fast_path": not scorer.use_forward, "dev": DEV}
save_activity(FEAT_OUT, res, text_emb, config)

print(res["stats"])
print("failed:", res["failed"][:5])
v_ex = next(iter(res["clip_emb"]))
print(f"example: video {v_ex} -> clip embeddings {res['clip_emb'][v_ex].shape}")
!ls -lh {FEAT_OUT}

In [ ]:
# 8. Build the curves and check the signal: is each curve higher inside the GT than outside?
import numpy as np
from vidsearch.xclip_features import load_activity, activity_curves
from vidsearch import curves as cv

act = load_activity(FEAT_OUT)
for r in recs:                                     # durations, in case the jsonl has none
    if "duration" not in r and r["vid"] in act["videos"]:
        r["duration"] = act["videos"][r["vid"]]["duration"]

CURVES = {}
if FRAME_FEAT_DIR:
    print("reading SigLIP features from", FRAME_FEAT_DIR)
    S = cv.load_frame_curves(FRAME_FEAT_DIR, recs)
    if S:
        CURVES["frame"] = S
CURVES["activity_cond"] = activity_curves(act, recs, "cond")    # full X-CLIP (video-conditioned text)
CURVES["activity_plain"] = activity_curves(act, recs, "plain")  # ablation: plain cosine, no prompting

SIGNAL = {}
for name, C in CURVES.items():
    pct, n = cv.inside_vs_outside(C, val)
    SIGNAL[name] = pct
    print(f"{name:15s} inside > outside on val: {pct}% of {n} queries (50% = no signal) | curves: {len(C)}")

In [ ]:
# 9. Plot frame vs activity curves against the GT (an "opposite actions" pair first, if there is one)
import matplotlib.pyplot as plt, textwrap

pairs_val = cv.find_opposite_pairs(val)
examples = [pairs_val[0][1], pairs_val[0][2]] if pairs_val else []
pool = [r for r in val if r["qid"] not in {e["qid"] for e in examples}]
rng = np.random.default_rng(0)
examples += [pool[i] for i in rng.choice(len(pool), size=min(4 - len(examples), len(pool)), replace=False)]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, r in zip(axes.flat, examples):
    a = CURVES["activity_cond"].get(r["qid"])
    if a is not None:
        ax.plot(np.arange(len(a)) + 0.5, cv.zscore(a), color="tab:orange", lw=2, label="activity a(t), X-CLIP")
    if "frame" in CURVES and r["qid"] in CURVES["frame"]:
        s = CURVES["frame"][r["qid"]]
        ax.plot(np.arange(len(s)) + 0.5, cv.zscore(s), color="tab:blue", alpha=0.8, label="frame s(t), SigLIP")
    ax.axvspan(r["start"], r["end"], color="green", alpha=0.2, label="GT")
    ax.set_title(textwrap.shorten(r["query"], 60) + f"  [{r['vid']}]", fontsize=10)
    ax.set_xlabel("seconds"); ax.set_ylabel("z-score")
axes.flat[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RES_OUT / "activity_curves_examples.png", dpi=120); plt.show()

In [ ]:
# 10. Tune each curve x proposal method on VAL (CPU, about a minute in the full run)
import pandas as pd

BEST = {}
for name, C in CURVES.items():
    for method in ("threshold", "windows"):
        p, rows = cv.tune(C, val, method)
        BEST[(name, method)] = (p, rows[0])
        pd.DataFrame(rows).to_csv(RES_OUT / f"val_tuning_{name}_{method}.csv", index=False)

display(pd.DataFrame([{"evidence": n, "method": m, "R1@0.5": row["R1@0.5"], "R1@0.7": row["R1@0.7"],
                       "R5@0.5": row["R5@0.5"], "mIoU": row["mIoU"], "best params": p}
                      for (n, m), (p, row) in BEST.items()]))

ACT = max(["activity_cond", "activity_plain"],
          key=lambda n: max(BEST[(n, m)][1]["R1@0.5"] for m in ("threshold", "windows")))
print("activity scoring chosen on val:", ACT)

In [ ]:
# 11. TEST with the frozen val settings, log every row
import json
TEST_ROWS, PER_QUERY = [], {}
for (name, method), (p, _) in BEST.items():
    m, per = cv.evaluate(cv.predict(CURVES[name], test, method, p), test)
    TEST_ROWS.append({"evidence": name, "method": method, **m, "params": p})
    PER_QUERY[f"{name}_{method}"] = per
    cv.log_row(RESULTS_CSV, f"{name}_{method}", "test", m, p, model=MODEL, dev=DEV)

cols = ["evidence", "method", "R1@0.3", "R1@0.5", "R1@0.7", "R5@0.5", "R5@0.7", "mIoU", "n_queries"]
display(pd.DataFrame(TEST_ROWS)[cols])
json.dump(PER_QUERY, open(RES_OUT / "per_query_top1_iou_test.json", "w"))   # for Step 6
print("Bar to beat: prior R1@0.5 = 13.8 | notebook 01: frame threshold 40.4, frame windows 32.8")

In [ ]:
# 12. Can each model tell opposite actions apart? (e.g. opening vs closing the same window)
pairs = cv.find_opposite_pairs(val + test)
print(f"{len(pairs)} opposite-action pairs on the same video with separate moments, e.g.:")
for kind, a, b in pairs[:3]:
    print(f"  [{kind}] '{a['query']}'  vs  '{b['query']}'")
DIRECTION = {}
for name, C in CURVES.items():
    pct, n = cv.direction_accuracy(C, pairs)
    DIRECTION[name] = pct
    print(f"{name:15s} higher on its own moment than on the opposite one: {pct}% of {n} (50% = cannot tell)")

In [ ]:
# 13. Efficiency numbers for the report, and a summary file
import time
q0 = next(r for r in recs if r["vid"] == vid0)
pw = BEST[(ACT, "windows")][0]
t = time.perf_counter()
for _ in range(50):
    cv.window_proposals(CURVES[ACT][q0["qid"]], **pw)
prop_ms = 1000 * (time.perf_counter() - t) / 50
score_ms = scorer.query_latency_ms(clips0["frames"], clips0["idx"], q0["query"])

EFF = {**act["stats"],
       "query_scoring_ms_gpu": round(score_ms, 2),          # text encode + prompting + similarity
       "proposal_ms_cpu": round(prop_ms, 2),
       "feature_storage_MB": round(sum(f.stat().st_size for f in FEAT_OUT.iterdir()) / 1e6, 1)}
for k, v in EFF.items():
    print(f"{k:28s} {v}")

json.dump({"model": MODEL, "dev": DEV, "signal_inside_gt_pct": SIGNAL, "direction_pct": DIRECTION,
           "activity_variant": ACT, "best_params": {f"{n}_{m}": p for (n, m), (p, _) in BEST.items()},
           "test": TEST_ROWS, "efficiency": EFF},
          open(RES_OUT / "step3_summary.json", "w"), indent=2, default=str)

In [ ]:
# 14. Preview only (VAL): does mixing s(t) and a(t) help? Proper tuning is Step 4.
if "frame" in CURVES:
    out = []
    for alpha in (1.0, 0.75, 0.5, 0.25, 0.0):                  # alpha = weight on the frame curve
        F = {q: cv.fuse(CURVES["frame"][q], CURVES[ACT][q], alpha)
             for q in CURVES["frame"] if q in CURVES[ACT]}
        for method in ("threshold", "windows"):
            m, _ = cv.evaluate(cv.predict(F, val, method, BEST[("frame", method)][0]), val)
            out.append({"alpha": alpha, "method": method, "R1@0.5": m["R1@0.5"], "mIoU": m["mIoU"]})
    display(pd.DataFrame(out).pivot(index="alpha", columns="method", values=["R1@0.5", "mIoU"]))
else:
    print("SigLIP features not attached, so no fusion preview.")

### Full run
1. Set `DEV = False` in cell 3.
2. **Save Version → Save & Run All (Commit)**, then stop this interactive session (power icon) so it stops using GPU quota.
3. When the version finishes: **Output → New Dataset**, name it `vidsearch-activity`, and share it with your teammate.

The output should contain `features_activity/` (4 files: three `.npz` and `xclip_meta.json`) and `results/` (CSV tables, `step3_summary.json`, and the plot).